# Financial Sentiment Analysis — QLoRA + Pinecone RAG
**Model**: LLaMA-2-7B | **Technique**: QLoRA | **Dataset**: Financial Phrasebank

Run cells in order from top to bottom.

## Cell 1 — Mount Google Drive

In [ ]:
!git -C /content/financial-llm-sentiment pull

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/financial-llm/saved_model', exist_ok=True)
os.makedirs('/content/drive/MyDrive/financial-llm/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/financial-llm/logs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/financial-llm/cache', exist_ok=True)

print('Drive mounted and folders created.')

## Cell 2 — Set API Keys (from Colab Secrets)

In [ ]:
# Add your keys in Colab: left sidebar → 🔑 Secrets
# Keys needed: PINECONE_API_KEY, HF_TOKEN

from google.colab import userdata
import os

os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')
os.environ['HF_TOKEN']         = userdata.get('HF_TOKEN')

# Cache HuggingFace datasets to Drive so we don't re-download each session
os.environ['HF_DATASETS_CACHE'] = '/content/drive/MyDrive/financial-llm/cache'

print('API keys loaded.')

## Cell 3 — Clone Repo + Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/vkquanghd/financial-llm-sentiment.git'
REPO_DIR = '/content/financial-llm-sentiment'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

for folder in ['data', 'model', 'rag', 'app']:
    open(f'{folder}/__init__.py', 'w').close()
    print(f'✅ {folder}/__init__.py')

!pip install -q -r requirements.txt
print('Setup complete.')

import sys
if '/content/financial-llm-sentiment' not in sys.path:
    sys.path.insert(0, '/content/financial-llm-sentiment')
print('sys.path updated.')

## Cell 4 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU  : {gpu_name}')
    print(f'VRAM : {vram_gb:.1f} GB')
    print(f'BF16 : {torch.cuda.is_bf16_supported()}')
else:
    print('No GPU detected. Go to Runtime → Change runtime type → GPU (A100).')

## Cell 5 — Prepare Data (preview)

In [ ]:
import os
os.chdir('/content/financial-llm-sentiment')

for folder in ['data', 'model', 'rag', 'app']:
    open(f'{folder}/__init__.py', 'w').close()

print('Done.')

## Cell 6 — Train Model

In [ ]:
import sys
import importlib

# Remove cached failed imports
for key in list(sys.modules.keys()):
    if 'model' in key or 'data' in key or 'rag' in key or 'app' in key:
        del sys.modules[key]

# Try import again
from model.train import train
print('Import successful!')

In [ ]:
import sys
sys.path.insert(0, '/content/financial-llm-sentiment')

from model.train import train

trainer, tokenizer, rag_sentences = train()

## Cell 7 — Index Financial Sentences into Pinecone

In [ ]:
from rag.pinecone_utils import initialize_rag

# rag_sentences came from train() — reuse without reloading data
index, embed_model = initialize_rag(rag_sentences)

print(f'Pinecone index stats: {index.describe_index_stats()}')

## Cell 8 — Test Inference

In [ ]:
from model.inference import predict_sentiment, predict_with_rag

test_sentences = [
    'The company reported record profits this quarter.',
    'Markets remained flat amid economic uncertainty.',
    'Stock prices collapsed after the earnings miss.',
]

print('=== Without RAG ===')
for s in test_sentences:
    result = predict_sentiment(s, trainer.model, tokenizer)
    print(f'  [{result["label"].upper():8s}] {s}')

print('\n=== With RAG ===')
for s in test_sentences:
    result = predict_with_rag(s, trainer.model, tokenizer, index, embed_model)
    print(f'  [{result["label"].upper():8s}] {s}')
    print(f'  Context: {result["retrieved_context"][0][:80]}...')

## Cell 9 — Evaluate on Test Set

In [ ]:
from data.prepare_data import prepare_dataset
from model.evaluate import evaluate_on_testset

tokenized_dataset, _, _ = prepare_dataset()
results = evaluate_on_testset(trainer.model, tokenizer, tokenized_dataset['test'])

## Cell 10 — Attention Heatmap

In [ ]:

import importlib
import model.evaluate
importlib.reload(model.evaluate)
from model.evaluate import plot_attention_heatmap

plot_attention_heatmap(
    sentence  = 'The company reported record profits this quarter.',
    model     = trainer.model,
    tokenizer = tokenizer,
)

## Cell 11 — Benchmark Table

In [ ]:
from model.evaluate import build_benchmark_table

# Fill in the numbers after running each model
benchmark_results = {
    'BERT-base':   {'params_M': 110,  'time_min': 15, 'memory_gb': 2.1, 'f1': 0.0, 'accuracy': 0.0},
    'LLaMA-QLoRA': {'params_M': 7000, 'time_min': 45, 'memory_gb': 5.2, 'f1': results['f1_macro'], 'accuracy': results['accuracy']},
}

df = build_benchmark_table(benchmark_results)

## Cell 12 — Launch Gradio App

In [ ]:
import importlib
import app.gradio_app as app_module
importlib.reload(app_module)
from app.gradio_app import build_app

app_module.model       = trainer.model
app_module.tokenizer   = tokenizer
app_module.index       = index
app_module.embed_model = embed_model

gradio_app = build_app()
gradio_app.launch(share=True)

In [ ]:
import json
from google.colab import userdata

# Fix notebook
nb_path = "/content/financial-llm-sentiment/train_colab.ipynb"
with open(nb_path) as f:
    nb = json.load(f)
nb["metadata"].pop("widgets", None)
with open(nb_path, "w") as f:
    json.dump(nb, f, indent=1)

# Push
token = userdata.get("GITHUB_TC")
!git config --global user.email "vokhacquang2000@gmail.com"
!git config --global user.name "vkquanghd"
!git -C /content/financial-llm-sentiment remote set-url origin https://{token}@github.com/vkquanghd/financial-llm-sentiment.git
!git -C /content/financial-llm-sentiment add train_colab.ipynb
!git -C /content/financial-llm-sentiment commit -m "fix: remove broken widget metadata from notebook"
!git -C /content/financial-llm-sentiment push origin main